In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% ! important;}
div.cell.code_cell.rendered{width:100%}
div.input_prompt{padding:0px}
div.CodeMirror {font-family:Consolas ; font-size:12pt;}
div.text_cell_render.rendered_html {font-size:12pt;}
div.output {font-size:12pt; font-weight:bold}
div.input {font-family:Consolas ; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper {padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

# red wine 품질 등급 예측

```
1. 데이터셋 확보 & 전처리
    독립변수와 타겟변수 분리 -> 독립변수 스케일조정(StandardScaler) 
    -> 타겟변수 원핫인코딩(get_dummies)
2. 모델 구성(입력11, 출력?)
3. 모델학습과정 설정(다중분류로 설정)
4. 모델 학습(callbacks이용)
5. 모델 평가 - 그래프, 평가(테스트셋), 교차표/혼동행렬
6. 모델 저장/ 사용
```

In [3]:
import numpy as np
import pandas as pd # read_csv, get_dummies, crosstab
from sklearn.preprocessing import StandardScaler # , MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential, save_model, load_model
from tensorflow.keras.layers import Input, Dense, Dropout, LeakyReLU
from tensorflow.keras import metrics
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
import matplotlib.pyplot as plt

## 1. 데이터셋 확보 & 전처리
```
    독립변수와 타겟변수 분리 -> 독립변수 스케일조정(StandardScaler) 
    -> 타겟변수 원핫인코딩(get_dummies)
    
    Fixed acidity (고정 산도): 주로 타르타르산(tartaric acid)의 양.
    Volatile acidity (휘발성 산도): 주로 아세트산(acetic acid)의 양으로, 높을수록 불쾌한 식초 맛을 유발할 수 있습니다.
    Citric acid (구연산): 와인에 신선함과 풍미를 더합니다.
    Residual sugar (잔여 당분): 발효 후 남은 당분의 양.
    Chlorides (염화물): 와인 속 소금의 양.
    Free sulfur dioxide (유리 아황산가스): 미생물 성장과 산화를 방지하는 역할을 합니다.
    Total sulfur dioxide (총 아황산가스): 유리 및 결합 형태의 아황산가스 총량.
    Density (밀도): 당도가 높을수록 밀도가 높아지는 경향이 있습니다.
    pH: 산도 수준을 나타냅니다.
    Sulphates (황산염): SO2 수준에 기여하는 와인 첨가물입니다.
    Alcohol (알코올): 와인 속 알코올 함량.
    Quality (품질): 0(매우 나쁨)에서 10(매우 우수) 사이의 전문가 평가 점수 (실제 데이터셋의 범위는 3~8).
```

In [7]:
# 데이터 읽어오기
# np.loadtxt('data/winequality-red.csv', delimiter=';', skiprows=1)
# np.genfromtxt('data/winequality-red.csv', delimiter=';', skip_header=1)
redwine = pd.read_csv('./data/winequality-red.csv', sep=';')
redwine.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [10]:
# 타겟변수의 균형
redwine['quality'].value_counts()

5    681
6    638
7    199
4     53
8     18
3     10
Name: quality, dtype: int64

In [13]:
# 독립변수, 타겟변수 분리
X_redwine = redwine.iloc[:,:-1].values # to_numpy와 유사, numpy배열로
y_redwine = redwine.iloc[:,-1] # 3,4,5,6,7,8
X_redwine.shape, y_redwine.shape

((1599, 11), (1599,))

In [15]:
# 독립변수 x의 스케일 조정
scaler = StandardScaler()
scaler.fit(X_redwine)
scaled_X_redwine = scaler.transform(X_redwine)
# 다중분류를 위한 타겟변수의 원핫인코딩 -> numpy 배열로 변환
Y_redwine = pd.get_dummies(y_redwine).to_numpy()

In [16]:
# 독립변수      &        타겟변수
scaled_X_redwine.shape, Y_redwine.shape

((1599, 11), (1599, 6))